In [10]:
# Uncomment if you dont have xgboost installed on your machine.
# %pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [1]:
# Import Required Libraries
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from xgboost import plot_tree
from sklearn.metrics import accuracy_score
from sklearn import tree
from sklearn.multioutput import MultiOutputClassifier
from PIL import Image


In [2]:
# loading the data file
df = pd.read_excel("../Processed_Data.xlsx")

In [3]:
# Print the length of the DataFrame 'df' to display the number of rows in the dataset
print("Dataset length: ", len(df))

# Print the shape of the DataFrame 'df', which includes the number of rows and columns in a tuple
print("Dataset Shape: ", df.shape)


Dataset length:  7000
Dataset Shape:  (7000, 13)


In [4]:
# Function to create the target Label
def createTarget(df):
    # Create a new column 'Labels' in the DataFrame and initialize it with a list of 8 zeros for each row
    df['Labels'] = [[0, 0, 0, 0, 0, 0, 0, 0] for i in range(0, len(df))]

    # Define the target columns you want to extract values from
    target_columns = ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']

    # Create an empty list called 'label' to store values for each row
    label = []

    # Loop through each row in the DataFrame
    for i in range(0, len(df)):
        # Loop through the target columns
        for target in target_columns:
            # Append the value from the current target column for the current row to the 'label' list
            label.append(df.loc[i, target])

        # Update the 'Labels' column for the current row with the 'label' list
        df.at[i, 'Labels'] = label

        # Reset the 'label' list for the next row
        label = []

In [5]:
# Call the 'createTarget' function to modify the 'df' DataFrame
createTarget(df)

# Display the Dataframe
df.head()

,Patient ID,Patient Age,Patient Sex,Filename,Diagnosis,N,D,G,C,A,H,M,O,Labels
0,0,69,Female,0_left.jpg,cataract,0,0,0,1,0,0,0,0,"[0, 0, 0, 1, 0, 0, 0, 0]"
1,1,57,Male,1_left.jpg,normal fundus,1,0,0,0,0,0,0,0,"[1, 0, 0, 0, 0, 0, 0, 0]"
2,2,42,Male,2_left.jpg,"laser spot,moderate non proliferative retinopathy",0,1,0,0,0,0,0,1,"[0, 1, 0, 0, 0, 0, 0, 1]"
3,3,66,Male,3_left.jpg,normal fundus,1,0,0,0,0,0,0,0,"[1, 0, 0, 0, 0, 0, 0, 0]"
4,4,53,Male,4_left.jpg,macular epiretinal membrane,0,0,0,0,0,0,0,1,"[0, 0, 0, 0, 0, 0, 0, 1]"


In [6]:
# Dropping ID column and original label columns
outcome = df.drop(['Patient ID', 'N', 'D', 'G', 'C', 'A', 'H', 'M', 'O'], axis=1)
#  Displaying the new dataframe
outcome.head(20)

,Patient Age,Patient Sex,Filename,Diagnosis,Labels
0,69,Female,0_left.jpg,cataract,"[0, 0, 0, 1, 0, 0, 0, 0]"
1,57,Male,1_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]"
2,42,Male,2_left.jpg,"laser spot,moderate non proliferative retinopathy","[0, 1, 0, 0, 0, 0, 0, 1]"
3,66,Male,3_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]"
4,53,Male,4_left.jpg,macular epiretinal membrane,"[0, 0, 0, 0, 0, 0, 0, 1]"
5,50,Female,5_left.jpg,moderate non proliferative retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]"
6,60,Male,6_left.jpg,macular epiretinal membrane,"[0, 0, 0, 0, 0, 0, 0, 1]"
7,60,Female,7_left.jpg,drusen,"[0, 0, 0, 0, 0, 0, 0, 1]"
8,59,Male,8_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]"
9,54,Male,9_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]"


In [7]:
# Path to your image dataset
image_dataset_path = "../ocular-disease-recognition-odir5k/ODIR-5K/ODIR-5K/Training images"

In [8]:
# Initialize the lists to hold the feature vectors and labels
features = []
labels = []

# Process the images and labels
for index, row in df.iterrows():
    # Open the image
    image_path = os.path.join(image_dataset_path, row['Filename'])
    with Image.open(image_path) as img:
        img = img.resize((128, 128))  # Resize the image
        img_array = np.array(img)  # Convert to numpy array
        img_array = img_array.flatten()  # Flatten the array

    # Append the image (as a flattened array) to the features
    features.append(img_array)
    # Append the label to the labels list
    labels.append(row['Labels'])

# Convert lists to numpy arrays
features = np.array(features)
labels = np.array(labels)

In [9]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.3, random_state=42)

# Initialize the XGBoost Classifier
model = MultiOutputClassifier(XGBClassifier(random_state=42))

# Train the model
model.fit(X_train, y_train)

# Predict the labels on the test set
y_pred = model.predict(X_test)

XGBoostError: [19:53:50] C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0750514818a16474a-1\xgboost\xgboost-ci-windows\src\common\io.h:232: bad_malloc: Failed to allocate 8467670736 bytes.

In [36]:
# Calculate the accuracy of a machine learning model by comparing its predictions 'y_pred'
# to the true labels 'y_test' from the test set
accuracy = accuracy_score(y_test, y_pred)

# Print the accuracy of the XG Boost  classifier on the test set as a percentage with two decimal places
print(f'Accuracy of the XGBoost classifier on the test set: {accuracy:.2%}')

Accuracy of the XGBoost classifier on the test set: 26.24%


In [26]:
#Min Max Scaling Age
age_scaler = MinMaxScaler()
df['Patient Age'] = age_scaler.fit_transform(df[['Patient Age']])

In [27]:
# Initialize the lists to hold the feature vectors and labels
age_features = []
labels = []

# Process the images and labels:
for index, row in df.iterrows():
    # Open the image
    img_path = os.path.join(image_dataset_path, row['Filename'])
    with Image.open(img_path) as img:
        
        img = img.resize((128, 128))  #resizing 
        img_array = np.array(img)  # Convert to numpy array
        img_array = img_array.flatten()  # Flatten the image array
    
    # Append the age to the image array.
    patient_age = np.array(row['Patient Age'])
    patient_age = np.expand_dims(patient_age, axis=0)
    feature_with_age = np.concatenate((img_array, patient_age)) #concatenating both features

    # Append the feature_with_age array to the features list
    age_features.append(feature_with_age)
    
    # Append the label (which is already a list) to the labels list
    labels.append(row['Labels'])

# Convert lists to numpy arrays
features = np.array(features)
labels = np.array(labels)



In [28]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(age_features, labels, test_size=0.3, random_state=42)

# Initialize the XGBoost Classifier
model_with_age = MultiOutputClassifier(XGBClassifier(random_state=42))

# Train the model
model_with_age.fit(X_train, y_train)

# Predict the labels on the test set
y_pred = model_with_age.predict(X_test)


In [29]:
# Calculate the accuracy of a machine learning model by comparing its predictions 'y_pred'
# to the true labels 'y_test' from the test set
accuracy = accuracy_score(y_test, y_pred)

# Print the accuracy of the XG Boost  classifier on the test set as a percentage with two decimal places
print(f'Accuracy of the XGBoost classifier on the test set: {accuracy:.2%}')

Accuracy of the XGBoost classifier on the test set: 25.19%
